# 🎙️ Piper TTS — Entraînement voix Guy Chapelier (KITT VF)

**Avant de commencer :**
1. Menu `Exécution` → `Modifier le type d'exécution` → choisir **GPU T4**
2. Uploader `dataset_guy_chapelier.zip` via le panneau Fichiers (icône dossier à gauche) → glisser-déposer
3. Puis `Exécution` → `Tout exécuter`

Durée estimée : **30–60 minutes**

In [ ]:
# ── Cellule 1 : Vérification GPU ──────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip())
if not result.stdout:
    print('⚠️  Pas de GPU ! Va dans Exécution → Modifier le type d\'exécution → GPU T4')

In [ ]:
# ── Cellule 2 : Installation dépendances ──────────────────────────────────
!apt-get install -y espeak-ng espeak-ng-data libespeak-ng-dev -q
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# piper-phonemize (roue précompilée x86_64)
!pip install -q piper-phonemize

# piper_train depuis le repo officiel
!pip install -q 'git+https://github.com/rhasspy/piper.git#egg=piper_train&subdirectory=src/python'

print('✅ Dépendances installées')

In [ ]:
# ── Cellule 3 : Extraction du dataset ────────────────────────────────────
import os, zipfile

ZIP_PATH = '/content/dataset_guy_chapelier.zip'
DATASET_DIR = '/content/dataset'
MODEL_DIR = '/content/model_guy'

os.makedirs(MODEL_DIR, exist_ok=True)

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(
        '❌ dataset_guy_chapelier.zip introuvable !\n'
        'Upload-le via le panneau Fichiers (icône dossier à gauche)'
    )

print('📦 Extraction...')
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(DATASET_DIR)

# Vérifie la structure
wav_count = len([f for f in os.listdir(f'{DATASET_DIR}/wavs') if f.endswith('.wav')])
csv_lines = sum(1 for _ in open(f'{DATASET_DIR}/metadata.csv', encoding='utf-8'))
print(f'✅ {wav_count} fichiers WAV | {csv_lines} lignes metadata.csv')

In [ ]:
# ── Cellule 4 : Téléchargement checkpoint de base (fr_FR-tom) ────────────
# On part d'un modèle français existant pour le fine-tuner → BEAUCOUP plus rapide
import urllib.request

CHECKPOINT_DIR = '/content/checkpoint_base'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

base_url = 'https://huggingface.co/rhasspy/piper-voices/resolve/main/fr/fr_FR/tom/medium'
files = [
    ('fr_FR-tom-medium.onnx',       f'{base_url}/fr_FR-tom-medium.onnx'),
    ('fr_FR-tom-medium.onnx.json',  f'{base_url}/fr_FR-tom-medium.onnx.json'),
]

for fname, url in files:
    dest = f'{CHECKPOINT_DIR}/{fname}'
    if not os.path.exists(dest):
        print(f'⬇️  {fname}...')
        urllib.request.urlretrieve(url, dest)
    else:
        print(f'✅ {fname} déjà présent')

print('✅ Checkpoint de base prêt')

In [ ]:
# ── Cellule 5 : Prétraitement du dataset ─────────────────────────────────
PREPROCESSED_DIR = f'{MODEL_DIR}/preprocessed'

!python3 -m piper_train.preprocess \
    --language fr \
    --input-dir {DATASET_DIR} \
    --output-dir {PREPROCESSED_DIR} \
    --dataset-format ljspeech \
    --sample-rate 22050

# Vérifie
import glob
phoneme_files = glob.glob(f'{PREPROCESSED_DIR}/**/*.pt', recursive=True)
print(f'✅ Prétraitement terminé — {len(phoneme_files)} fichiers .pt générés')

In [ ]:
# ── Cellule 6 : ENTRAÎNEMENT ──────────────────────────────────────────────
# 200 epochs = ~45 min sur T4, qualité très correcte
# Augmente max_epochs à 500 pour meilleure qualité (mais ~2h)

EPOCHS = 200

print(f'🚀 Démarrage entraînement — {EPOCHS} epochs...')
print('   (une barre de progression va apparaître)')

!python3 -m piper_train \
    --dataset-dir {PREPROCESSED_DIR} \
    --accelerator gpu \
    --devices 1 \
    --batch-size 32 \
    --validation-split 0.05 \
    --num-test-examples 5 \
    --max_epochs {EPOCHS} \
    --output-dir {MODEL_DIR}/lightning_logs \
    --precision 16-mixed \
    --checkpoint-epochs 50

In [ ]:
# ── Cellule 7 : Export ONNX ───────────────────────────────────────────────
# Trouve le dernier checkpoint
import glob, os

checkpoints = sorted(
    glob.glob(f'{MODEL_DIR}/lightning_logs/**/*.ckpt', recursive=True)
)
if not checkpoints:
    raise FileNotFoundError('❌ Aucun checkpoint trouvé — vérifier cellule 6')

last_ckpt = checkpoints[-1]
print(f'📌 Checkpoint utilisé : {os.path.basename(last_ckpt)}')

ONNX_OUT = '/content/guy_chapelier.onnx'

!python3 -m piper_train.export_onnx \
    --checkpoint {last_ckpt} \
    --output {ONNX_OUT}

# Copie le config JSON (nécessaire pour Piper)
import shutil, glob
config_src = glob.glob(f'{PREPROCESSED_DIR}/**/config.json', recursive=True)
if config_src:
    shutil.copy(config_src[0], '/content/guy_chapelier.onnx.json')
    print('✅ config.json copié')

size = os.path.getsize(ONNX_OUT) / 1024 / 1024
print(f'✅ ONNX exporté : {ONNX_OUT} ({size:.1f} Mo)')

In [ ]:
# ── Cellule 8 : Test rapide de la voix ───────────────────────────────────
!pip install -q piper-tts

TEST_PHRASE = "Bonjour Michael. Tous mes systèmes sont opérationnels. Je suis prêt."

!echo "{TEST_PHRASE}" | python3 -m piper \
    --model /content/guy_chapelier.onnx \
    --output_file /content/test_guy.wav

# Écoute dans Colab
from IPython.display import Audio, display
display(Audio('/content/test_guy.wav'))
print('🔊 Écoute le résultat ci-dessus !')

In [ ]:
# ── Cellule 9 : Téléchargement ────────────────────────────────────────────
# Télécharge les 2 fichiers nécessaires pour le Jetson
from google.colab import files

print('📥 Téléchargement des fichiers...')
files.download('/content/guy_chapelier.onnx')
files.download('/content/guy_chapelier.onnx.json')
files.download('/content/test_guy.wav')

print()
print('✅ Fini ! Tu as 3 fichiers :')
print('   guy_chapelier.onnx       → modèle voix')
print('   guy_chapelier.onnx.json  → config')
print('   test_guy.wav             → test audio')
print()
print('Envoie-les à Claude pour qu\'il les installe sur le Jetson !')